In [1]:
import pandas as pd

In [2]:
df = pd.read_csv(r"C:\Users\udaym\Downloads\Teen_Mental_Health_Dataset.csv")

In [3]:
df

,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label
0,14,male,7.9,Instagram,7.4,2.9,3.01,1.5,low,2,2,1,0
1,19,female,1.9,TikTok,8.0,2.9,3.22,0.8,high,8,1,10,0
2,17,female,1.3,Instagram,7.6,0.5,3.92,0.0,high,2,4,2,0
3,15,male,7.4,TikTok,6.9,1.6,3.48,0.8,medium,1,7,9,0
4,15,female,4.7,Both,4.9,3.0,2.37,1.4,medium,3,5,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,18,female,6.8,Instagram,6.6,2.0,2.76,1.0,low,3,4,4,0
1196,16,male,2.3,Both,8.0,1.9,2.12,0.4,high,7,4,4,0
1197,14,female,1.7,Both,8.7,0.7,3.98,0.8,high,1,1,1,0
1198,15,male,3.9,Both,8.5,2.1,3.19,0.6,high,7,9,9,0


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age                       1200 non-null   int64  
 1   gender                    1200 non-null   str    
 2   daily_social_media_hours  1200 non-null   float64
 3   platform_usage            1200 non-null   str    
 4   sleep_hours               1200 non-null   float64
 5   screen_time_before_sleep  1200 non-null   float64
 6   academic_performance      1200 non-null   float64
 7   physical_activity         1200 non-null   float64
 8   social_interaction_level  1200 non-null   str    
 9   stress_level              1200 non-null   int64  
 10  anxiety_level             1200 non-null   int64  
 11  addiction_level           1200 non-null   int64  
 12  depression_label          1200 non-null   int64  
dtypes: float64(5), int64(5), str(3)
memory usage: 140.4 KB


In [12]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.metrics import accuracy_score, precision_score

In [8]:
transformer = ColumnTransformer([
    ('ohe', OneHotEncoder(sparse_output=False, drop='first'), ['gender', 'platform_usage']),
    ('ordinal_enc', OrdinalEncoder(), ['social_interaction_level'])
], remainder='passthrough')

In [9]:
decision_pipe = Pipeline([
    ('transormer', transformer),
    ('model', DecisionTreeClassifier())
])

In [10]:
random_f_pipe = Pipeline([
    ('transormer', transformer),
    ('model', RandomForestClassifier())
])

In [11]:
X =df.drop(columns=['depression_label'])
y = df['depression_label']

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [18]:
models = {'decision_tree': decision_pipe, 'random_forest': random_f_pipe}

In [19]:
for name, model in  models.items():
    model.fit(X_train, y_train)
    
    print(f"Accuracy: {accuracy_score(y_test, model.predict(X_test))}\nPrecision: {precision_score(y_test, model.predict(X_test))}")

Accuracy: 1.0
Precision: 1.0
Accuracy: 0.9777777777777777
Precision: 1.0


In [20]:
bagging_pipe = Pipeline([
    ('transormer', transformer),
    ('model', BaggingClassifier())
])

In [23]:
for name, model in  models.items():
    bagging_pipe = Pipeline([
    ('transormer', transformer),
    ('model', BaggingClassifier(estimator=model['model']))
    ])
    bagging_pipe.fit(X_train, y_train)
    
    print(f"For: {name}\n\nAccuracy: {accuracy_score(y_test, bagging_pipe.predict(X_test))}\nPrecision: {precision_score(y_test, bagging_pipe.predict(X_test))}")


For: decision_tree

Accuracy: 0.9916666666666667
Precision: 1.0
For: random_forest

Accuracy: 0.975
Precision: 0.0


E:\class_repos\DataScience_evening\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
